In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "common.py").exists():
    AI_DIR = cwd
elif (cwd.parent / "common.py").exists():
    AI_DIR = cwd.parent
elif (cwd / "AI" / "common.py").exists():
    AI_DIR = cwd / "AI"
else:
    raise FileNotFoundError("Cannot locate AI/common.py")

if str(AI_DIR) not in sys.path:
    sys.path.insert(0, str(AI_DIR))

import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from common import FEATURE_COLUMNS, MODEL_PATH, fetch_dataset_frame, validate_feature_columns


In [2]:
df = fetch_dataset_frame()
if df.empty:
    raise ValueError("Dataset is empty. Cannot train model.")

print("Raw samples:", len(df))
if "device_id" in df.columns:
    print("Samples by device:")
    print(df["device_id"].value_counts())

df.head()


Raw samples: 124
Samples by device:
device_id
room_1    124
Name: count, dtype: int64


,delta_gas,device_id,gas,gas_relative,hour,humidity,label,month,temperature,timestamp
0,-18,room_1,422,0.59712,12,49.82634,0,3,32.32441,1774156139
1,-40,room_1,382,0.56656,12,49.89986,0,3,32.29675,1774156140
2,3,room_1,385,0.59660,12,49.91064,0,3,32.29904,1774156141
3,5,room_1,390,0.62924,12,49.90816,0,3,32.29580,1774156142
4,-11,room_1,379,0.63621,12,49.81947,0,3,32.31201,1774156143


In [3]:
required_columns = FEATURE_COLUMNS + ["label"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.dropna(subset=required_columns).copy()
validate_feature_columns(df)

print("Clean samples:", len(df))
print("Label distribution:")
print(df["label"].value_counts().sort_index())


Clean samples: 124
Label distribution:
label
0    29
1    28
3    67
Name: count, dtype: int64


In [4]:
X = df[FEATURE_COLUMNS]
y = df["label"]

stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 99
Test size: 25


In [5]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)

print("Accuracy:", accuracy)


Accuracy: 0.92


In [8]:
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")


Model saved to D:\VS Code\ESP32\GAS_SENSOR\AI\models\gas_model.pkl
